# VLM-DENTAL: Rules + Skills

Post-training, inference-only harness experiments. Use the fixed **dev** split; the official validation/test split is prohibited. This notebook never launches SFT/GRPO or changes the frozen protocol. Run in an isolated Kaggle/Colab session after your checkpoint is saved. Skills retain the existing tool workflow and explicit source references.

## 1. Isolated checkout
Apply the delivered harness patches to your repository and push them yourself first. Set REPO_REF to your reviewed harness commit. This creates a separate checkout; it does not pull over running training files.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
REPO_URL = "https://github.com/rezaxr14/VLM-DENTAL.git"
REPO_REF = "main"  # or the exact reviewed commit containing this patch
if not Path("dental_agent/harness/notebook_eval.py").is_file():
    parent = Path("/kaggle/working" if Path("/kaggle").exists() else "/content" if Path("/content").exists() else Path.cwd())
    work = parent / "VLM-DENTAL-EVALUATION"
    if not work.exists():
        if len(REPO_REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REPO_REF):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(work)], check=True)
            subprocess.run(["git", "-C", str(work), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
            subprocess.run(["git", "-C", str(work), "checkout", "--detach", "FETCH_HEAD"], check=True)
        else:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(work)], check=True)
    os.chdir(work)
assert Path("dental_agent/harness/notebook_eval.py").is_file(), "Use the checkout containing this patch"
IS_KAGGLE = Path("/kaggle").exists()
IS_COLAB = Path("/content").exists() and not IS_KAGGLE
print("Checkout:", Path.cwd())

## 2. Dependencies and credentials
Prefer a fresh notebook runtime. Existing training environments are not modified. Enable installation here only in this isolated runtime. Dependency versions remain those declared by the repository.

In [ ]:
INSTALL_DEPENDENCIES = True  # install only in this isolated evaluation runtime
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[api,dev]"], check=True)
from dotenv import load_dotenv
load_dotenv()
for key in ("HF_TOKEN", "HF_ARTIFACT_REPO", "DENTEX_IMAGES_REPO"):
    if os.environ.get(key): continue
    try:
        if IS_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            value = UserSecretsClient().get_secret(key)
        elif IS_COLAB:
            from google.colab import userdata
            value = userdata.get(key)
        else: value = None
        if value: os.environ[key] = value
    except Exception: pass
print("Artifacts:", os.environ.get("HF_ARTIFACT_REPO", "Reza-Nadimi/vlm-dental-models"))

## 3. Profile, checkpoint and fixed dev inputs
Hardware profile determines experiment identity and numerical runtime settings. Never silently substitute another checkpoint. A GRPO checkpoint should point at its step/final folder; its grpo_policy adapter is selected by the existing evaluator. No optimizer state is needed for inference.

In [ ]:
from dental_agent.harness.notebook_eval import NotebookEvaluations
import json
# Profiles are visible here. Aliases: rtx4090 -> rtx4090_qlora; rtxpro6000_48gb -> workstation48_lora.
EVAL_PROFILES = {
    "dual_t4": {
        "tag": "dual_t4_fp16",
        "precision": "qlora",
        "compute": "fp16",
        "attention": "sdpa",
        "kv": "on",
        "loader": "staged",
        "threads": 1,
        "letterbox": False
    },
    "rtx4090_qlora": {
        "tag": "rtx4090_bf16",
        "precision": "qlora",
        "compute": "bf16",
        "attention": "auto",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False
    },
    "workstation48_lora": {
        "tag": "workstation48_lora_bf16",
        "precision": "bf16",
        "compute": "auto",
        "attention": "auto",
        "kv": "auto",
        "loader": "standard",
        "threads": 0,
        "letterbox": False
    },
    "tpu": {
        "tag": "tpu_bf16",
        "precision": "bf16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "off",
        "loader": "standard",
        "threads": 0,
        "letterbox": True
    },
    "dual_t4_fp16": {
        "tag": "dual_t4_unquantized_fp16",
        "precision": "fp16",
        "compute": "auto",
        "attention": "sdpa",
        "kv": "on",
        "loader": "staged",
        "threads": 1,
        "letterbox": False
    }
}
# Existing result repository; checkpoints still come from HF_ARTIFACT_REPO.
EVAL_RESULTS_REPO = "Reza-Nadimi/vlm-dental-evaluations"
EVAL_RESULTS_REPO_TYPE = None  # Detect the existing model or dataset repository; never create one.
EVAL_TRACK = "with_tools"  # with_tools, no_tools, or all
print(json.dumps(EVAL_PROFILES, indent=2))
EVAL_TARGET_PROFILES = "dual_t4"  # any key above, a list of keys, or "all"

EVAL_STAGES = "all"
EVAL_SELECTED_K = 4
MODEL_ID = None  # existing resolver prefers local/attached weights
DATA_DIR = "data"
DEV_SPLIT = "data/splits/dentex_dev.json"
RUN_EVALUATIONS = True
MAX_NEW_TOKENS = 16384
LIMIT = None  # full fixed dev cohort

In [ ]:
harness_evaluations = NotebookEvaluations(track=EVAL_TRACK, profiles=EVAL_PROFILES, results_repo=EVAL_RESULTS_REPO, results_repo_type=EVAL_RESULTS_REPO_TYPE, target_profiles=EVAL_TARGET_PROFILES,
    kinds=["rules_skills"], stages=EVAL_STAGES, selected_k=EVAL_SELECTED_K,
    model_id=MODEL_ID, data_dir=DATA_DIR, dev_split=DEV_SPLIT, split="dev")
harness_evaluations.list()

## 4. Input and contract audit (no model weights loaded)
Attach your training images/data to this isolated session, or set DATA_DIR to the local training-data directory. No held-out test files are read.

In [ ]:
from dental_agent.harness.rules import validate_implementations
from dental_agent.harness.skills import SkillLibrary
validate_implementations()
print(SkillLibrary().index())

## 5. Paired dev evaluation
The four conditions share the exact checkpoint, image cohort, 16K budget, seed and tool implementation. Saved episodes resume; completed cohorts skip model loading.

In [ ]:
HARNESS_EVAL_STATUS = harness_evaluations.run(execute=RUN_EVALUATIONS)
print(json.dumps(HARNESS_EVAL_STATUS, indent=2))

## 6. Report and skill-load audit
Pooled exact F1 and invented-findings rate use shared matching. Undefined rates are null, never invented as zero. Inspect format failures alongside clinical metrics.

In [ ]:
from dental_agent.harness.notebook_eval import summarize_inventory
HARNESS_SUMMARY = summarize_inventory(HARNESS_EVAL_STATUS)
print(json.dumps(HARNESS_SUMMARY, indent=2))

## 7. Protocol amendment — review only
The amendment below is a proposal. This notebook does not apply it or edit the frozen protocol.

In [ ]:
print(Path('docs/harness/proposals/frozen_protocol_amendment.patch').read_text())


## 8. Saved results
Each result bundle is automatically saved under `Reza-Nadimi/vlm-dental-evaluations/evaluations/dentex/`. No repository is created and no Git push occurs.

In [ ]:
for item in HARNESS_EVAL_STATUS:
    print(item.get("hub_path", item.get("reason", item["status"])))